# K-Means Clustering on the Iris Dataset — From Scratch

_Generated: 2025-10-22T18:04:14.701240Z_

This notebook implements **K-Means** with **pure NumPy** (random init → assignment → centroid update) and applies it to the classic **Iris** dataset. It includes standardization, **PCA (via SVD)** for 2D visualization, inertia tracking, an **elbow plot**, and a **silhouette score** implementation. We also compute a simple **cluster purity** against the (unused) ground-truth species labels for reference.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib scikit-learn

## 1) Imports & Helper Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

def standardize(X, eps=1e-12):
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd = np.where(sd < eps, 1.0, sd)
    Xs = (X - mu) / sd
    return Xs, mu, sd

def pca_svd(X, n_components=2):
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    comps = Vt[:n_components]
    Z = Xc @ comps.T
    explained_var = (S**2) / (len(X)-1)
    evr = explained_var[:n_components] / explained_var.sum()
    return Z, comps, evr

## 2) Load Iris Data & Standardize

In [ ]:
iris = datasets.load_iris()
X = iris.data.astype(float)
y = iris.target
target_names = iris.target_names
feature_names = iris.feature_names

print("X shape:", X.shape, "| classes:", list(target_names))
X_std, mu, sd = standardize(X)
Z, comps, evr = pca_svd(X_std, n_components=2)
print("PCA EVR (first two):", evr)

## 3) K-Means Implementation (NumPy only)

In [ ]:
def kmeans(X, k=3, max_iter=300, tol=1e-6, seed=SEED, n_init=10):
    n, d = X.shape
    best_inertia = np.inf
    best_centroids = None
    best_labels = None
    best_hist = None
    rng_local = np.random.default_rng(seed)

    for run in range(n_init):
        idx = rng_local.choice(n, size=k, replace=False)
        C = X[idx].copy()
        labels = np.zeros(n, dtype=int)
        hist = []
        prev_inertia = np.inf
        for it in range(1, max_iter+1):
            d2 = np.sum((X[:, None, :] - C[None, :, :])**2, axis=2)
            labels = np.argmin(d2, axis=1)
            inertia = float(np.sum(np.min(d2, axis=1)))
            hist.append((it, inertia))
            if abs(prev_inertia - inertia) < tol:
                break
            prev_inertia = inertia
            C_new = C.copy()
            for j in range(k):
                pts = X[labels==j]
                if len(pts) > 0:
                    C_new[j] = pts.mean(axis=0)
                else:
                    C_new[j] = X[rng_local.integers(0, n)]
            C = C_new
        if inertia < best_inertia:
            best_inertia = inertia
            best_centroids = C.copy()
            best_labels = labels.copy()
            best_hist = hist
    return best_centroids, best_labels, best_inertia, best_hist

## 4) Run K-Means (k=3)

In [ ]:
k = 3
C, labels, inertia, hist = kmeans(X_std, k=k, max_iter=300, tol=1e-7, n_init=20, seed=SEED)
print(f"k={k} | inertia={inertia:.4f} | iterations={len(hist)}")

## 5) Convergence: Inertia vs Iteration

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.plot([it for it,_ in hist], [val for _,val in hist])
plt.xlabel("Iteration"); plt.ylabel("Inertia (sum of squared distances)")
plt.title("K-Means Convergence (best run)")
plt.tight_layout(); plt.show()

## 6) Visualization in 2D (PCA projection)

In [ ]:
C_proj = (C - X_std.mean(axis=0)) @ comps.T
fig = plt.figure(figsize=(6,5))
for j in range(k):
    pts = Z[labels==j]
    plt.scatter(pts[:,0], pts[:,1], label=f"cluster {j}", s=24)
plt.scatter(C_proj[:,0], C_proj[:,1], marker='X', s=150)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("K-Means Clusters (PCA 2D)")
plt.legend(); plt.tight_layout(); plt.show()

## 7) Elbow Method (k=1..6)

In [ ]:
ks = range(1,7)
inertias = []
for kk in ks:
    _, _, inh, _ = kmeans(X_std, k=kk, max_iter=200, tol=1e-6, n_init=10, seed=SEED)
    inertias.append(inh)

fig = plt.figure(figsize=(6,4))
plt.plot(list(ks), inertias, marker='o')
plt.xlabel("k"); plt.ylabel("Inertia")
plt.title("Elbow Curve")
plt.xticks(list(ks))
plt.tight_layout(); plt.show()

print("Inertias:", list(zip(ks, np.round(inertias, 2))))

## 8) Silhouette Score (from scratch)

In [ ]:
def pairwise_distances(X):
    X2 = np.sum(X*X, axis=1, keepdims=True)
    D2 = X2 + X2.T - 2*X@X.T
    D2[D2<0] = 0.0
    return np.sqrt(D2)

def silhouette_score_labels(X, labels):
    n = len(labels)
    D = pairwise_distances(X)
    sil = np.zeros(n)
    for i in range(n):
        ci = labels[i]
        mask_same = (labels == ci)
        mask_same[i] = False
        if np.any(mask_same):
            a = np.mean(D[i, mask_same])
        else:
            a = 0.0
        b = np.inf
        for c in np.unique(labels):
            if c == ci: continue
            mask_c = (labels == c)
            if np.any(mask_c):
                b = min(b, np.mean(D[i, mask_c]))
        denom = max(a, b) if max(a,b) > 0 else 1.0
        sil[i] = (b - a) / denom
    return float(np.mean(sil)), sil

sil_mean, sil_all = silhouette_score_labels(X_std, labels)
print(f"Silhouette score (k={k}): {sil_mean:.3f}")

## 9) Cluster Purity vs. True Species (reference only)

In [ ]:
def cluster_purity(y_true, labels, k):
    purity_sum = 0
    mapping = {}
    for j in range(k):
        idx = np.where(labels==j)[0]
        if len(idx)==0:
            mapping[j] = None
            continue
        vals, counts = np.unique(y_true[idx], return_counts=True)
        maj = vals[np.argmax(counts)]
        purity_sum += np.max(counts)
        mapping[j] = int(maj)
    purity = purity_sum / len(y_true)
    return purity, mapping

purity, mapping = cluster_purity(y, labels, k)
print(f"Purity (majority mapping): {purity:.3f}")
print("Cluster → species mapping:", {j: (mapping[j] if mapping[j] is None else target_names[mapping[j]]) for j in range(k)})

## 10) Save Artifacts & Download

In [ ]:
import os, json as _json
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/kmeans_iris_from_scratch.npz",
         X=X, X_std=X_std, y=y, labels=labels, centroids=C, inertia=np.array([inertia]),
         mu=mu, sd=sd, pcs=comps, Z=Z, silhouette=np.array([sil_mean]))

with open("artifacts/summary.json","w") as f:
    _json.dump({
        "k": int(k),
        "inertia": float(inertia),
        "silhouette": float(sil_mean),
        "purity": float(purity),
        "cluster_to_species": {int(j): (None if mapping[j] is None else target_names[mapping[j]]) for j in range(k)}
    }, f, indent=2)

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 11) Exercises & Extensions

- Implement **k-means++** seeding and compare convergence/inertia.
- Explore different `k` and compare **silhouette** and **purity**.
- Try **Mahalanobis** distance with a pooled covariance estimate.
- Implement **soft k-means** (EM for spherical Gaussians).
- Visualize with **3D PCA** or **t-SNE**.